# Notebook 04 — GNN Training: TLS Functional State Classifier

**Input**: `tls_graphs.pt` (915 spatial subgraphs from nb03)  
**Output**: trained model checkpoint + predictions on all 915 TLS regions  

## Architecture summary

```
Spot-level graph  →  GAT × 2  →  DiffPool (k=5 niches)
                  →  DenseGAT × 2  →  DiffPool (k=2 regions)
                  →  DenseGAT  →  mean-pool  →  MLP  →  {immunogenic, tolerogenic}
```

Parameters tuned from nb03 graph statistics:  
- `in_dim = 61` (50 PCA + 11 functional scores)  
- `k_niche = 5` (p25 nodes = 22 → 22//4 = 5)  
- `k_region = 2` (k_niche // 3 = 1 → max(2,1) = 2)  

Class imbalance: 668 immunogenic vs 122 tolerogenic → weighted CE loss.

In [ ]:
import os, sys, json, random, time
from pathlib import Path

# Add project root to path so src.* imports work
PROJECT_ROOT = Path('/home/gpeng/projects/spatial_transcriptom/tls_functional_score')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for cluster jobs
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, accuracy_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch_geometric.loader import DataLoader

from src.models.gnn import TLSFunctionalGNN
from src.training.losses import TLSTrainingLoss
from src.training.evaluate import evaluate

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Paths and Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Cluster data paths (written by nb03)
CLUSTER_DATA = Path('/.mounts/labs/gsiprojects/gsi/gsiusers/gpeng/dev/st/data')
GRAPHS_PT   = CLUSTER_DATA / 'processed' / 'tls_graphs.pt'
ARCH_CFG    = CLUSTER_DATA / 'splits' / 'arch_config.json'
SPLITS_JSON = CLUSTER_DATA / 'splits' / 'sample_splits.json'

# Local output dir
CKPT_DIR  = PROJECT_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print('Graphs file:', GRAPHS_PT)
print('Exists:', GRAPHS_PT.exists())

## 2. Load Graphs and Split

In [ ]:
print('Loading graphs ...')
t0 = time.time()
graphs = torch.load(GRAPHS_PT, weights_only=False)
print(f'Loaded {len(graphs)} graphs in {time.time()-t0:.1f}s')

# Show graph attributes
g0 = graphs[0]
print('\nGraph attributes:', [k for k in g0.keys()])
print('x shape:', g0.x.shape)
print('edge_index shape:', g0.edge_index.shape)
print('y:', g0.y, '  split:', g0.split if hasattr(g0, "split") else "N/A")

# Split by g.split attribute
train_graphs = [g for g in graphs if g.split == 'train']
val_graphs   = [g for g in graphs if g.split == 'val']
test_graphs  = [g for g in graphs if g.split == 'test']

print(f'\nSplit sizes:')
print(f'  Train: {len(train_graphs)} graphs')
print(f'  Val:   {len(val_graphs)} graphs')
print(f'  Test:  {len(test_graphs)} graphs')

In [ ]:
# Label inventory
LABEL_NAMES = {0: 'immunogenic', 1: 'tolerogenic', -1: 'uncertain'}

def label_counts(glist):
    from collections import Counter
    return Counter(int(g.y) for g in glist)

for split_name, glist in [('train', train_graphs), ('val', val_graphs), ('test', test_graphs)]:
    counts = label_counts(glist)
    labeled = counts.get(0, 0) + counts.get(1, 0)
    print(f'{split_name:6s}: immuno={counts.get(0,0):4d}  tolero={counts.get(1,0):4d}  uncertain={counts.get(-1,0):4d}  total={len(glist)}')

# Class weights for imbalanced CE loss
# Using inverse-frequency weighting on training set
train_counts = label_counts(train_graphs)
n_immuno = train_counts.get(0, 1)
n_tolero = train_counts.get(1, 1)
n_labeled = n_immuno + n_tolero

w_immuno = n_labeled / (2 * n_immuno)
w_tolero = n_labeled / (2 * n_tolero)
class_weights = torch.tensor([w_immuno, w_tolero], dtype=torch.float32).to(DEVICE)
print(f'\nClass weights: immunogenic={w_immuno:.3f}, tolerogenic={w_tolero:.3f}')

## 3. Data Loaders

In [ ]:
# Load arch config from nb03
if ARCH_CFG.exists():
    with open(ARCH_CFG) as f:
        arch = json.load(f)
    print('arch_config.json keys:', list(arch.keys()))
    IN_DIM   = arch['in_dim']
    K_NICHE  = arch.get('k_niche_clusters', arch.get('k_niche', 5))
    K_REGION = arch.get('k_region_clusters', arch.get('k_region', 2))
    print('Arch config from nb03:')
    print(f'  in_dim={IN_DIM}, k_niche={K_NICHE}, k_region={K_REGION}')
else:
    IN_DIM   = graphs[0].x.shape[1]
    K_NICHE  = 5
    K_REGION = 2
    print(f'arch_config.json not found — using fallback: in_dim={IN_DIM}, k_niche={K_NICHE}, k_region={K_REGION}')

BATCH_SIZE = 16  # smaller than 32 since some graphs are large

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
all_loader   = DataLoader(graphs,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

## 4. Model Instantiation

In [ ]:
HIDDEN  = 128
HEADS   = 4
DROPOUT = 0.2

model = TLSFunctionalGNN(
    in_dim           = IN_DIM,
    hidden           = HIDDEN,
    n_classes        = 2,
    n_niche_clusters = K_NICHE,
    n_region_clusters= K_REGION,
    heads            = HEADS,
    dropout          = DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: TLSFunctionalGNN')
print(f'  in_dim={IN_DIM}, hidden={HIDDEN}, k_niche={K_NICHE}, k_region={K_REGION}')
print(f'  Total parameters: {n_params:,}')
print(model)

## 5. Training Setup

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 100
PATIENCE     = 15
WARMUP_FRAC  = 0.05

# Loss weights
TASK_W        = 1.0
CONTRASTIVE_W = 0.1
AUX_W         = 0.01

# Focal loss settings (replaces weighted CE)
# gamma=2 suppresses easy immunogenic examples, keeps gradient on hard tolerogenic
USE_FOCAL   = True
FOCAL_GAMMA = 2.0

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=1, eta_min=1e-5)

loss_fn = TLSTrainingLoss(
    task_weight        = TASK_W,
    domain_weight      = 0.0,
    contrastive_weight = CONTRASTIVE_W,
    aux_weight         = AUX_W,
    n_classes          = 2,
    class_weights      = class_weights,
    use_focal          = USE_FOCAL,
    focal_gamma        = FOCAL_GAMMA,
)

print('Optimizer:', optimizer.__class__.__name__)
print('Scheduler: CosineAnnealingWarmRestarts T_0=50')
print(f'Task loss: {"FocalLoss(gamma=" + str(FOCAL_GAMMA) + ")" if USE_FOCAL else "CrossEntropyLoss"}')
print(f'Loss weights: task={TASK_W}, contrastive={CONTRASTIVE_W}, aux={AUX_W}')
print(f'Class weights: immunogenic={w_immuno:.3f}, tolerogenic={w_tolero:.3f}')

## 6. Training Loop

In [ ]:
def warmup_lr(optimizer, step, warmup_steps, base_lr):
    """Linear warmup for the first `warmup_steps` steps."""
    if step < warmup_steps:
        scale = (step + 1) / max(warmup_steps, 1)
        for pg in optimizer.param_groups:
            pg['lr'] = base_lr * scale


def train_one_epoch(model, loader, optimizer, loss_fn, device, step, warmup_steps):
    model.train()
    totals = {k: 0.0 for k in ['total', 'task', 'contrastive', 'aux']}
    n_batches = 0

    for batch in loader:
        batch = batch.to(device)
        warmup_lr(optimizer, step, warmup_steps, LR)

        optimizer.zero_grad()
        logits, aux_loss = model(batch.x, batch.edge_index, batch.batch)
        embeddings = model.get_embeddings(batch.x, batch.edge_index, batch.batch)

        losses = loss_fn(
            task_logits  = logits,
            task_labels  = batch.y,
            embeddings   = embeddings,
            aux_loss     = aux_loss,
        )

        losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        for k in ['task', 'contrastive', 'aux']:
            totals[k] += losses[k].item()
        totals['total'] += losses['total'].item()
        n_batches += 1
        step += 1

    avg = {k: v / max(n_batches, 1) for k, v in totals.items()}
    return avg, step


@torch.no_grad()
def evaluate_loader(model, loader, device):
    """Returns metrics dict + raw (probs, labels) arrays."""
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch.x, batch.edge_index, batch.batch)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        labels = batch.y.cpu().numpy()
        mask   = labels >= 0
        if mask.sum() > 0:
            all_probs.append(probs[mask])
            all_labels.append(labels[mask])

    if not all_probs:
        return {'auc_roc': 0.0, 'auc_pr': 0.0, 'f1_macro': 0.0, 'accuracy': 0.0}, np.array([]), np.array([])

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {}
    try:    metrics['auc_roc'] = roc_auc_score(y_true, y_prob)
    except: metrics['auc_roc'] = 0.0
    try:    metrics['auc_pr']  = average_precision_score(y_true, y_prob)
    except: metrics['auc_pr']  = 0.0
    metrics['f1_macro']  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['accuracy']  = accuracy_score(y_true, y_pred)
    return metrics, y_prob, y_true


print('Functions defined. Starting training ...')

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
WARMUP_STEPS = int(WARMUP_FRAC * EPOCHS * len(train_loader))
print(f'Warmup steps: {WARMUP_STEPS} ({WARMUP_FRAC*100:.0f}% of {EPOCHS*len(train_loader)} total)')

best_val_auc   = 0.0
best_epoch     = 0
patience_ctr   = 0
global_step    = 0
history        = []

CKPT_BEST = CKPT_DIR / 'best_model.pt'

t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()

    train_losses, global_step = train_one_epoch(
        model, train_loader, optimizer, loss_fn,
        DEVICE, global_step, WARMUP_STEPS
    )
    scheduler.step(epoch - 1)   # update LR (post-warmup)

    val_metrics, _, _ = evaluate_loader(model, val_loader, DEVICE)

    lr_now = optimizer.param_groups[0]['lr']
    ep_time = time.time() - t_ep

    row = {
        'epoch'    : epoch,
        'train_loss'  : train_losses['total'],
        'train_task'  : train_losses['task'],
        'val_auc_roc' : val_metrics['auc_roc'],
        'val_auc_pr'  : val_metrics['auc_pr'],
        'val_f1'      : val_metrics['f1_macro'],
        'val_acc'     : val_metrics['accuracy'],
        'lr'          : lr_now,
    }
    history.append(row)

    print(
        f'Epoch {epoch:03d}/{EPOCHS} | '
        f'loss={train_losses["total"]:.4f} (task={train_losses["task"]:.4f}) | '
        f'val_auc={val_metrics["auc_roc"]:.4f} '
        f'val_ap={val_metrics["auc_pr"]:.4f} '
        f'val_f1={val_metrics["f1_macro"]:.4f} | '
        f'lr={lr_now:.6f} | {ep_time:.1f}s',
        flush=True
    )

    if val_metrics['auc_roc'] > best_val_auc:
        best_val_auc = val_metrics['auc_roc']
        best_epoch   = epoch
        patience_ctr = 0
        torch.save({
            'epoch'      : epoch,
            'state_dict' : model.state_dict(),
            'optimizer'  : optimizer.state_dict(),
            'val_auc'    : best_val_auc,
            'arch'       : dict(in_dim=IN_DIM, hidden=HIDDEN,
                                k_niche=K_NICHE, k_region=K_REGION),
        }, CKPT_BEST)
        print(f'  >> Best model saved (val_auc={best_val_auc:.4f})', flush=True)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (best was epoch {best_epoch})', flush=True)
            break

total_time = time.time() - t_start
print(f'\nTraining complete in {total_time/60:.1f} min')
print(f'Best val AUC-ROC: {best_val_auc:.4f} at epoch {best_epoch}')

## 7. Training Curves

In [ ]:
df_hist = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(df_hist['epoch'], df_hist['train_loss'],  label='Total loss', lw=2)
ax.plot(df_hist['epoch'], df_hist['train_task'],  label='Task loss',  lw=2, linestyle='--')
ax.axvline(best_epoch, color='red', lw=1, linestyle=':', label=f'Best (ep {best_epoch})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training Loss'); ax.legend()

ax = axes[1]
ax.plot(df_hist['epoch'], df_hist['val_auc_roc'], label='AUC-ROC', lw=2)
ax.plot(df_hist['epoch'], df_hist['val_auc_pr'],  label='AUC-PR',  lw=2)
ax.axvline(best_epoch, color='red', lw=1, linestyle=':')
ax.axhline(best_val_auc, color='green', lw=1, linestyle=':', label=f'Best={best_val_auc:.3f}')
ax.set_xlabel('Epoch'); ax.set_ylabel('AUC'); ax.set_title('Validation AUC'); ax.legend()

ax = axes[2]
ax.plot(df_hist['epoch'], df_hist['val_f1'],  label='F1 macro', lw=2)
ax.plot(df_hist['epoch'], df_hist['val_acc'], label='Accuracy',  lw=2)
ax.axvline(best_epoch, color='red', lw=1, linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score'); ax.set_title('Validation Metrics'); ax.legend()

plt.tight_layout()
fig.savefig(CKPT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
df_hist.to_csv(CKPT_DIR / 'training_history.csv', index=False)
print('Saved: training_curves.png, training_history.csv')

## 8. Evaluation on Val and Test Sets (Best Checkpoint)

In [ ]:
# Load best checkpoint
ckpt = torch.load(CKPT_BEST, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]} (val_auc={ckpt["val_auc"]:.4f})')

val_metrics,  val_probs,  val_labels  = evaluate_loader(model, val_loader,  DEVICE)
test_metrics, test_probs, test_labels = evaluate_loader(model, test_loader, DEVICE)

print('\n=== Val set ===' )
for k, v in val_metrics.items():
    print(f'  {k}: {v:.4f}')

print('\n=== Test set ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row_idx, (split_name, probs, labels, metrics) in enumerate([
    ('Val',  val_probs,  val_labels,  val_metrics),
    ('Test', test_probs, test_labels, test_metrics),
]):
    # ROC curve
    fpr, tpr, _ = roc_curve(labels, probs)
    ax = axes[row_idx, 0]
    ax.plot(fpr, tpr, lw=2, label=f'AUC={metrics["auc_roc"]:.3f}')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{split_name} ROC Curve'); ax.legend()

    # PR curve
    prec, rec, _ = precision_recall_curve(labels, probs)
    ax = axes[row_idx, 1]
    ax.plot(rec, prec, lw=2, label=f'AP={metrics["auc_pr"]:.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{split_name} PR Curve'); ax.legend()

    # Confusion matrix
    y_pred = (probs >= 0.5).astype(int)
    cm = confusion_matrix(labels, y_pred)
    ax = axes[row_idx, 2]
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Immuno', 'Tolero'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'{split_name} Confusion Matrix')

plt.tight_layout()
fig.savefig(CKPT_DIR / 'evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evaluation_plots.png')

## 9. Score Distribution and Calibration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (split_name, probs, labels) in zip(axes, [
    ('Val',  val_probs,  val_labels),
    ('Test', test_probs, test_labels),
]):
    immuno = probs[labels == 0]
    tolero = probs[labels == 1]
    bins = np.linspace(0, 1, 25)
    ax.hist(immuno, bins=bins, alpha=0.6, color='#e64b35', label=f'Immunogenic (n={len(immuno)})', density=True)
    ax.hist(tolero, bins=bins, alpha=0.6, color='#4dbbd5', label=f'Tolerogenic (n={len(tolero)})',  density=True)
    ax.axvline(0.5, color='k', linestyle='--', lw=1.5, label='threshold=0.5')
    ax.set_xlabel('P(tolerogenic)'); ax.set_ylabel('Density')
    ax.set_title(f'{split_name}: Score Distribution'); ax.legend()

plt.tight_layout()
fig.savefig(CKPT_DIR / 'score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: score_distributions.png')

## 10. Inference on All 915 Graphs — Save Predictions

In [ ]:
@torch.no_grad()
def run_inference(model, loader, device):
    """Returns DataFrame with per-TLS predictions + embeddings."""
    model.eval()
    records = []
    all_embeddings = []

    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch.x, batch.edge_index, batch.batch)
        emb       = model.get_embeddings(batch.x, batch.edge_index, batch.batch)
        probs     = torch.softmax(logits, dim=1)

        all_embeddings.append(emb.cpu())

        # Resolve per-graph attributes (PyG stores lists for non-tensor attrs,
        # but tensor attrs like cluster_id need .item() to get a plain int)
        cluster_ids = batch.cluster_id if hasattr(batch, 'cluster_id') else [-1] * len(batch)
        sample_ids  = batch.sample_id  if hasattr(batch, 'sample_id')  else [''] * len(batch)
        splits      = batch.split      if hasattr(batch, 'split')      else [''] * len(batch)
        label_names = batch.label_name if hasattr(batch, 'label_name') else [''] * len(batch)

        for i in range(len(batch)):
            cid = cluster_ids[i]
            # If cluster_id is a tensor element, convert to plain Python int
            if hasattr(cid, 'item'):
                cid = cid.item()
            records.append({
                'cluster_id'          : cid,
                'sample_id'           : sample_ids[i],
                'split'               : splits[i],
                'label_name'          : label_names[i],
                'y_true'              : int(batch.y[i]),
                'p_immunogenic'       : float(probs[i, 0]),
                'p_tolerogenic'       : float(probs[i, 1]),
                'tls_functional_score': float(probs[i, 1]),
                'pred_label'          : int(probs[i].argmax()),
            })

    df = pd.DataFrame(records)
    emb_tensor = torch.cat(all_embeddings, dim=0)
    return df, emb_tensor


print('Running inference on all 915 TLS graphs ...')
pred_df, embeddings = run_inference(model, all_loader, DEVICE)
print(f'Predictions shape: {pred_df.shape}')
print(f'Embeddings shape: {embeddings.shape}')
pred_df.head(10)

In [ ]:
# Save predictions CSV
pred_csv = CLUSTER_DATA / 'processed' / 'tls_predictions.csv'
pred_df.to_csv(pred_csv, index=False)
print(f'Saved predictions -> {pred_csv}')

# Save embeddings tensor
emb_pt = CLUSTER_DATA / 'processed' / 'tls_embeddings.pt'
torch.save(embeddings, emb_pt)
print(f'Saved embeddings  -> {emb_pt}')

# Save a local copy of predictions for quick access
local_csv = PROJECT_ROOT / 'checkpoints' / 'tls_predictions.csv'
pred_df.to_csv(local_csv, index=False)
print(f'Local copy         -> {local_csv}')

# Distribution of predicted scores
print('\n=== Score distribution across all 915 TLS regions ===')
print(pred_df['tls_functional_score'].describe().round(4))

labeled_df = pred_df[pred_df['y_true'] >= 0]
print(f'\n=== Prediction accuracy on labeled set ({len(labeled_df)} graphs) ===')
correct = (labeled_df['pred_label'] == labeled_df['y_true']).sum()
print(f'Accuracy: {correct}/{len(labeled_df)} = {correct/len(labeled_df):.3f}')

## 11. UMAP of TLS Embeddings

In [ ]:
try:
    import umap
    has_umap = True
except ImportError:
    has_umap = False
    print('umap-learn not installed — skipping UMAP. Install with: pip install umap-learn')

if has_umap:
    print('Running UMAP on TLS embeddings ...')
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.3)
    emb_2d = reducer.fit_transform(embeddings.numpy())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    LABEL_COLORS = {0: '#e64b35', 1: '#4dbbd5', -1: '#cccccc'}
    LABEL_NAMES  = {0: 'Immunogenic', 1: 'Tolerogenic', -1: 'Uncertain'}

    # Color by label
    ax = axes[0]
    for lbl, color in LABEL_COLORS.items():
        mask = pred_df['y_true'].values == lbl
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color,
                   label=LABEL_NAMES[lbl], s=15, alpha=0.7, linewidths=0)
    ax.set_title('Colored by true label')
    ax.legend(markerscale=2, fontsize=8)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

    # Color by predicted score
    ax = axes[1]
    sc = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                    c=pred_df['tls_functional_score'].values,
                    cmap='RdBu_r', s=15, alpha=0.7, linewidths=0, vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax, label='P(tolerogenic)')
    ax.set_title('Colored by TLS functional score')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

    # Color by sample/data split
    ax = axes[2]
    SPLIT_COLORS = {'train': '#999999', 'val': '#f39c12', 'test': '#8e44ad'}
    for split_name, color in SPLIT_COLORS.items():
        mask = pred_df['split'].values == split_name
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color,
                   label=split_name, s=15, alpha=0.7, linewidths=0)
    ax.set_title('Colored by data split')
    ax.legend(markerscale=2, fontsize=8)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

    plt.tight_layout()
    fig.savefig(CKPT_DIR / 'umap_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: umap_embeddings.png')

    # Save 2D coordinates
    pred_df['umap1'] = emb_2d[:, 0]
    pred_df['umap2'] = emb_2d[:, 1]
    pred_df.to_csv(pred_csv, index=False)        # update with UMAP coords
    pred_df.to_csv(local_csv, index=False)

## 12. Summary for Notebook 05

In [ ]:
print('=' * 55)
print('=== Summary for notebook 05 ===')
print('=' * 55)
print(f'Best checkpoint  : {CKPT_BEST}')
print(f'  epoch={ckpt["epoch"]}, val_auc={ckpt["val_auc"]:.4f}')
print()
print('Val metrics:')
for k, v in val_metrics.items():
    print(f'  {k}: {v:.4f}')
print()
print('Test metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')
print()
print(f'Predictions CSV  : {pred_csv}')
print(f'Embeddings PT    : {emb_pt}')
print(f'Training history : {CKPT_DIR}/training_history.csv')
print()
print('Next: nb05 -- clinical validation')
print('  - Kaplan-Meier curves: high vs. low TLS functional score')
print('  - ICB response prediction (AUC on responder/non-responder labels)')
print('  - Cross-cancer validation (PDAC, multi-cancer datasets)')

## 13. 5-Fold Sample-Level Cross-Validation

With only 2 samples per val/test in the main split, single-split metrics are noisy and
val AUC is biased (optimized against one dominant sample). 5-fold CV at the sample level
gives a more honest estimate of generalisation: each fold holds out ~5 samples (~2 with
tolerogenic TLS) and retrains from scratch.

**Paper-ready numbers come from here, not from Section 8.**

Expected runtime: ~10-15 min on V100/A100 (80 epochs x 5 folds).

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

# ── Build sample inventory ─────────────────────────────────────────────────────
sample_info = {}
for g in graphs:
    sid = g.sample_id
    if sid not in sample_info:
        sample_info[sid] = {'n_tolero': 0, 'n_immuno': 0, 'n_graphs': 0}
    sample_info[sid]['n_graphs'] += 1
    y = int(g.y)
    if y == 1:
        sample_info[sid]['n_tolero'] += 1
    elif y == 0:
        sample_info[sid]['n_immuno'] += 1

all_samples = np.array(sorted(sample_info.keys()))
has_tolero  = np.array([sample_info[s]['n_tolero'] > 0 for s in all_samples], dtype=int)

print(f'Total samples: {len(all_samples)}')
print(f'  With tolerogenic TLS : {has_tolero.sum()}')
print(f'  Without tolerogenic  : {(has_tolero == 0).sum()}')
print()

for s in all_samples:
    info = sample_info[s]
    print(f'  {s}: immuno={info["n_immuno"]:3d}  tolero={info["n_tolero"]:3d}  '
          f'graphs={info["n_graphs"]:3d}  has_tol={info["n_tolero"]>0}')

# ── Fold definitions ──────────────────────────────────────────────────────────
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f'\n5-fold sample-level split (stratified by tolerogenic presence):')
for fold_idx, (tv_idx, test_idx) in enumerate(skf.split(all_samples, has_tolero)):
    tol_in_test = has_tolero[test_idx].sum()
    print(f'  Fold {fold_idx+1}: test={len(test_idx)} samples '
          f'({tol_in_test} with tolero) | train+val={len(tv_idx)} samples')

In [ ]:
# ── CV hyperparameters ────────────────────────────────────────────────────────
CV_EPOCHS = 80     # fixed for all folds -- AUC from 3-4 tolero is too noisy to stop on
CV_T0     = 40     # cosine restart period

cv_results = []
t_cv_start = time.time()

for fold_idx, (tv_idx, test_idx) in enumerate(skf.split(all_samples, has_tolero)):
    test_samples      = set(all_samples[test_idx])
    train_val_samples = all_samples[tv_idx]

    # ── Simple random 20% inner val (no tolero-rich guarantee) ───────────────
    # Fix A: removing the tolero-rich-in-val guarantee -- it steals training
    # signal from the minority class more than it helps early stopping quality.
    rng = np.random.RandomState(SEED + fold_idx)
    n_inner_val = max(2, len(train_val_samples) // 5)
    inner_val_idx   = rng.choice(len(train_val_samples), size=n_inner_val, replace=False)
    inner_val_set   = set(train_val_samples[inner_val_idx])
    inner_train_set = set(train_val_samples) - inner_val_set

    # Graph lists
    fold_train_g = [g for g in graphs if g.sample_id in inner_train_set]
    fold_val_g   = [g for g in graphs if g.sample_id in inner_val_set]
    fold_test_g  = [g for g in graphs if g.sample_id in test_samples]

    n_tol_test      = sum(1 for g in fold_test_g  if int(g.y) == 1)
    n_lab_test      = sum(1 for g in fold_test_g  if int(g.y) >= 0)
    n_tol_train     = sum(1 for g in fold_train_g if int(g.y) == 1)
    n_tol_inner_val = sum(1 for g in fold_val_g   if int(g.y) == 1)

    print(f'\n{"="*60}')
    print(f'Fold {fold_idx+1}/{N_FOLDS}')
    print(f'  train : {len(fold_train_g)} graphs ({len(inner_train_set)} samples, {n_tol_train} tolero)')
    print(f'  val   : {len(fold_val_g)} graphs ({len(inner_val_set)} samples, {n_tol_inner_val} tolero)')
    print(f'  test  : {len(fold_test_g)} graphs ({len(test_samples)} samples, '
          f'{n_tol_test}/{n_lab_test} tolero)')
    print(f'  test samples: {sorted(test_samples)}', flush=True)
    # Fix B: always fixed epochs -- no early stopping in CV
    print(f'  Training: fixed {CV_EPOCHS} epochs (no early stopping)', flush=True)

    if len(fold_train_g) == 0 or n_lab_test == 0:
        print('  Skipping: no training data or no labeled test graphs', flush=True)
        continue

    # Class weights for this fold
    fold_n_imm = sum(1 for g in fold_train_g if int(g.y) == 0)
    fold_n_tol = max(sum(1 for g in fold_train_g if int(g.y) == 1), 1)
    fold_n_lab = fold_n_imm + fold_n_tol
    fold_cw = torch.tensor([
        fold_n_lab / (2 * max(fold_n_imm, 1)),
        fold_n_lab / (2 * fold_n_tol),
    ], dtype=torch.float32).to(DEVICE)
    print(f'  class weights: immuno={fold_cw[0]:.3f}, tolero={fold_cw[1]:.3f}', flush=True)

    # DataLoaders
    fl_train = DataLoader(fold_train_g, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    fl_val   = DataLoader(fold_val_g,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    fl_test  = DataLoader(fold_test_g,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # Fresh model + optimizer
    fold_model = TLSFunctionalGNN(
        in_dim=IN_DIM, hidden=HIDDEN, n_classes=2,
        n_niche_clusters=K_NICHE, n_region_clusters=K_REGION,
        heads=HEADS, dropout=DROPOUT,
    ).to(DEVICE)
    fold_opt   = AdamW(fold_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    fold_sched = CosineAnnealingWarmRestarts(fold_opt, T_0=CV_T0, eta_min=1e-5)

    # Focal loss in every fold (Fix 2, unchanged)
    fold_loss = TLSTrainingLoss(
        task_weight=TASK_W, domain_weight=0.0,
        contrastive_weight=CONTRASTIVE_W, aux_weight=AUX_W,
        n_classes=2, class_weights=fold_cw,
        use_focal=True, focal_gamma=FOCAL_GAMMA,
    )

    # ── Fixed-epoch training (Fix B) ──────────────────────────────────────────
    fold_step  = 0
    fold_warmup = max(1, int(0.05 * CV_EPOCHS * len(fl_train)))
    t_fold = time.time()

    for ep in range(1, CV_EPOCHS + 1):
        _, fold_step = train_one_epoch(
            fold_model, fl_train, fold_opt, fold_loss, DEVICE, fold_step, fold_warmup
        )
        fold_sched.step(ep - 1)

    elapsed = time.time() - t_fold
    print(f'  Training done: {CV_EPOCHS} epochs in {elapsed:.0f}s', flush=True)

    # Evaluate on test fold
    test_m, test_prob, test_lbl = evaluate_loader(fold_model, fl_test, DEVICE)

    tol_recall = tol_prec = tol_f1 = float('nan')
    if n_tol_test > 0 and len(test_lbl) > 0:
        test_pred = (test_prob >= 0.5).astype(int)
        rep = classification_report(
            test_lbl, test_pred,
            target_names=['immunogenic', 'tolerogenic'],
            output_dict=True, zero_division=0
        )
        tol_recall = rep['tolerogenic']['recall']
        tol_prec   = rep['tolerogenic']['precision']
        tol_f1     = rep['tolerogenic']['f1-score']
        print(classification_report(
            test_lbl, test_pred,
            target_names=['immunogenic', 'tolerogenic'],
            digits=3, zero_division=0
        ), flush=True)

    print(f'  AUC-ROC={test_m["auc_roc"]:.4f}  AUC-PR={test_m["auc_pr"]:.4f}  '
          f'tolero_recall={tol_recall:.3f}  tolero_f1={tol_f1:.3f}', flush=True)

    cv_results.append({
        'fold'          : fold_idx + 1,
        'test_samples'  : ', '.join(sorted(test_samples)),
        'n_labeled'     : n_lab_test,
        'n_tol'         : n_tol_test,
        'n_tol_val'     : n_tol_inner_val,
        'auc_roc'       : test_m['auc_roc'],
        'auc_pr'        : test_m['auc_pr'],
        'f1_macro'      : test_m['f1_macro'],
        'accuracy'      : test_m['accuracy'],
        'tol_recall'    : tol_recall,
        'tol_precision' : tol_prec,
        'tol_f1'        : tol_f1,
    })

    del fold_model, fold_opt, fold_loss
    torch.cuda.empty_cache()

print(f'\nCV total time: {(time.time()-t_cv_start)/60:.1f} min')

In [ ]:
import math

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(CKPT_DIR / 'cv_results.csv', index=False)

# ── Per-fold table ─────────────────────────────────────────────────────────────
print('=== 5-Fold CV Results (per fold) ===\n')
cols = ['fold', 'n_labeled', 'n_tol', 'n_tol_val',
        'auc_roc', 'auc_pr', 'tol_recall', 'tol_precision', 'tol_f1', 'f1_macro', 'accuracy']
print(cv_df[cols].to_string(index=False,
      float_format=lambda x: f'{x:.3f}' if isinstance(x, float) and not math.isnan(x) else str(x)))

# ── Summary statistics ─────────────────────────────────────────────────────────
valid_cv    = cv_df[cv_df['n_tol'] > 0]
metric_cols = ['auc_roc', 'auc_pr', 'tol_recall', 'tol_precision', 'tol_f1', 'f1_macro', 'accuracy']

print(f'\n=== Mean +/- SD across {len(valid_cv)}/{len(cv_df)} folds with tolerogenic test data ===\n')
summary_rows = []
for col in metric_cols:
    vals = valid_cv[col].dropna()
    mean, std = vals.mean(), vals.std()
    print(f'  {col:16s}: {mean:.3f} +/- {std:.3f}')
    summary_rows.append({'metric': col, 'mean': mean, 'std': std, 'n_folds': len(vals)})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(CKPT_DIR / 'cv_summary.csv', index=False)
print('\nSaved: cv_results.csv, cv_summary.csv')

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

key_metrics = ['auc_roc', 'auc_pr', 'tol_recall']
titles      = ['AUC-ROC', 'AUC-PR (tolerogenic)', 'Tolerogenic Recall']
thresholds  = [None, None, 0.4]

for ax, metric, title, thresh in zip(axes, key_metrics, titles, thresholds):
    vals  = valid_cv[metric].dropna().values
    folds = valid_cv['fold'].values[:len(vals)]
    mean, std = vals.mean(), vals.std()

    ax.bar(folds, vals, color='#4dbbd5', alpha=0.7, width=0.6)
    ax.axhline(mean, color='black', lw=2, label=f'Mean={mean:.3f}')
    ax.axhline(mean + std, color='black', lw=1, linestyle='--', alpha=0.4)
    ax.axhline(mean - std, color='black', lw=1, linestyle='--', alpha=0.4,
               label=f'SD={std:.3f}')
    if thresh is not None:
        ax.axhline(thresh, color='#e64b35', lw=1.5, linestyle=':', label=f'Threshold={thresh}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric)
    ax.set_title(title + f'\n(fixed {CV_EPOCHS} epochs, focal loss)')
    ax.set_xticks(folds)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(CKPT_DIR / 'cv_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cv_results.png')

# ── Paper-ready summary ────────────────────────────────────────────────────────
auc_m = summary_df.loc[summary_df['metric'] == 'auc_roc',    'mean'].values[0]
auc_s = summary_df.loc[summary_df['metric'] == 'auc_roc',    'std'].values[0]
rec_m = summary_df.loc[summary_df['metric'] == 'tol_recall', 'mean'].values[0]
rec_s = summary_df.loc[summary_df['metric'] == 'tol_recall', 'std'].values[0]

print()
print('=== Paper-ready line ===')
print(f'5-fold CV AUC-ROC:            {auc_m:.3f} +/- {auc_s:.3f}')
print(f'5-fold CV tolerogenic recall: {rec_m:.3f} +/- {rec_s:.3f}')
print()
if rec_m < 0.4:
    print('WARNING: mean tolerogenic recall still below 0.4.')
    print('  Next steps: increase focal gamma, add graph augmentation,')
    print('  or review label quality before proceeding to notebook 05.')
else:
    print('Tolerogenic recall >= 0.4 -- proceed to notebook 05.')